# fintrack — 02 Data Exploration
**Purpose**: Ad-hoc exploration of transaction data via the fintrack API.  
**Version**: 1.0 — 2026-03-24  
Run cells top to bottom. Re-run individual cells as needed.

In [1]:
import os
import httpx
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

API_BASE  = f"http://{os.getenv('API_HOST','192.168.1.170')}:{os.getenv('API_PORT','8000')}"
EMAIL     = os.getenv('FINTRACK_EMAIL', 'your@email.com')
PASSWORD  = os.getenv('FINTRACK_PASSWORD', 'yourpassword')

print(f'API: {API_BASE}')

API: http://192.168.1.170:8000


In [2]:
# ── Authenticate ──────────────────────────────────────────────────────────────
r = httpx.post(f'{API_BASE}/api/v1/auth/login',
               json={'email': "mahajani6630@gmail.com", 'password': "Akamo@6630"}, timeout=10)
assert r.status_code == 200, f'Login failed: {r.text}'
TOKEN   = r.json()['access_token']
HEADERS = {'Authorization': f'Bearer {TOKEN}'}
print('Authenticated OK')

Authenticated OK


In [3]:
# ── Helper: call API and return DataFrame ─────────────────────────────────────
def api_get(path, params=None):
    r = httpx.get(f'{API_BASE}{path}', headers=HEADERS, params=params, timeout=15)
    r.raise_for_status()
    return r.json()

def transactions_df(page_size=200, **kwargs):
    """Fetch transactions as a DataFrame. Pass year=, month=, category= to filter."""
    params = {'password': PASSWORD, 'page_size': page_size, **kwargs}
    data   = api_get('/api/v1/transactions', params)
    df     = pd.DataFrame(data['items'])
    if not df.empty:
        df['date']   = pd.to_datetime(df['date'])
        df['amount'] = df['amount'].astype(float)
    print(f"Fetched {len(df)} of {data['total']} transactions (page 1 of {data['pages']})")
    return df

def all_transactions_df(**kwargs):
    """Fetch ALL transactions across all pages."""
    params = {'password': PASSWORD, 'page_size': 200, 'page': 1, **kwargs}
    first  = api_get('/api/v1/transactions', params)
    frames = [pd.DataFrame(first['items'])]
    for page in range(2, first['pages'] + 1):
        params['page'] = page
        frames.append(pd.DataFrame(api_get('/api/v1/transactions', params)['items']))
    df = pd.concat(frames, ignore_index=True)
    df['date']   = pd.to_datetime(df['date'])
    df['amount'] = df['amount'].astype(float)
    print(f"Fetched all {len(df)} transactions")
    return df

print('Helpers ready')

Helpers ready


## 1. Overview — totals by card provider

In [4]:
# Fetch all accounts
accounts = api_get('/api/v1/transactions/accounts', {'password': "Akamo@6630"})
pd.DataFrame(accounts)[['provider','label','member','source']]

HTTPStatusError: Client error '422 Unprocessable Entity' for url 'http://192.168.1.170:8000/api/v1/transactions/accounts?password=Akamo%406630'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/422

## 2. Category Summary

In [ ]:
cat = api_get('/api/v1/analytics/category-summary', {'year': 2025})
print(f"Grand Total:   ${cat['grand_total']:,.2f}")
print(f"Essential:     ${cat['essential_total']:,.2f}  ({cat['essential_pct']}%)")
print(f"Non-Essential: ${cat['nonessential_total']:,.2f}  ({cat['nonessential_pct']}%)")
print()
df_cat = pd.DataFrame(cat['categories'])
df_cat['total'] = df_cat['total'].round(2)
df_cat[['category','subcategory','total','txn_count','pct','is_essential']]

## 3. Monthly spend totals

In [5]:
trend = api_get('/api/v1/analytics/trend', {'year': 2025})
df_trend = pd.DataFrame(trend['months'])
df_trend[['month','total','txn_count','mom_delta','mom_pct']]

,month,total,txn_count,mom_delta,mom_pct
0,Jan 2025,15.71,1,0.00,0.0
1,Jan 2025,170.73,3,155.02,986.8
2,Jan 2025,80.81,2,-89.92,-52.7
3,Jan 2025,3474.03,2,3393.22,4199.0
4,Jan 2025,88.51,3,-3385.52,-97.5
...,...,...,...,...,...
252,Dec 2025,722.36,7,629.27,676.0
253,Dec 2025,61.63,1,-660.73,-91.5
254,Dec 2025,315.22,3,253.59,411.5
255,Dec 2025,122.08,2,-193.14,-61.3


## 4. Browse transactions by category

In [6]:
# Change category name to explore different categories
CATEGORY = 'Groceries'

df = transactions_df(category=CATEGORY, page_size=200)
print(f"\nTotal {CATEGORY}: ${df['amount'].sum():,.2f}  ({len(df)} transactions)")
df[['date','description','amount','category','source_file']].sort_values('amount', ascending=False)

HTTPStatusError: Client error '422 Unprocessable Entity' for url 'http://192.168.1.170:8000/api/v1/transactions?password=Akamo%406630&page_size=200&category=Groceries'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/422

## 5. Large expenses

In [7]:
THRESHOLD = 200
large = api_get('/api/v1/analytics/large-expenses',
                {'password': PASSWORD, 'threshold': THRESHOLD, 'year': 2025})
print(f"Expenses >= ${THRESHOLD}: {large['count']} transactions")
df_large = pd.DataFrame(large['items'])
df_large[['date','description','amount','category','subcategory','provider']]

HTTPStatusError: Client error '422 Unprocessable Entity' for url 'http://192.168.1.170:8000/api/v1/analytics/large-expenses?password=Akamo%406630&threshold=200&year=2025'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/422

## 6. Monthly pivot — categories as rows

In [8]:
pivot = api_get('/api/v1/analytics/monthly-pivot', {'year': 2025})
months = pivot['months']

rows = []
for r in pivot['rows']:
    row = {'Category': r['category'], 'Subcategory': r['subcategory'],
           'Essential': '✓' if r['is_essential'] else ''}
    for m in months:
        row[m] = r['months'].get(m, 0)
    row['Total'] = r['row_total']
    rows.append(row)

# Add column totals row
totals = {'Category': 'TOTAL', 'Subcategory': '', 'Essential': ''}
for m in months:
    totals[m] = pivot['col_totals'].get(m, 0)
totals['Total'] = pivot['grand_total']
rows.append(totals)

df_pivot = pd.DataFrame(rows)
# Format amounts
for col in months + ['Total']:
    df_pivot[col] = df_pivot[col].apply(lambda x: f'${x:,.0f}' if x else '-')

df_pivot.set_index(['Category','Subcategory']).style.set_caption('Monthly Spend by Category ($)')